# 09 — 2026 Scenario Forecasting

This notebook applies the selected forecasting models to generate 2026 monthly car registration forecasts under multiple macroeconomic scenarios.

The objective is to translate the model development work into scenario-based forecasts that can support business interpretation and market outlook analysis.

The forecasting workflow includes:

- Preparing 2026 macroeconomic and industry assumptions
- Reconstructing model-ready features for each forecast month
- Generating baseline, favorable, and unfavorable scenario forecasts
- Comparing outputs across selected models
- Producing a final hybrid forecast for interpretation and reporting

The final output is a monthly 2026 forecast table showing expected car registrations under each scenario.# Creating full 2026 macro model

In [ ]:
# =========================================================
#  1. Import Required Libraries
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# =========================================================
# Load datasets
# =========================================================
macro_df = pd.read_parquet("data/macro_df.parquet")
master_df_impute_eu = pd.read_parquet("data/master_df_impute_eu.parquet")



## Scenario Assumptions and Feature Reconstruction

This section defines the 2026 scenario engine used for forecasting.

The logic starts by taking 2025 EU-level macroeconomic and industry patterns as the base year, then adjusts them to create three 2026 scenarios:

- **Baseline**: central macroeconomic outlook
- **Favorable**: stronger growth, lower inflation/interest pressure, and improved energy conditions
- **Unfavorable**: weaker growth, higher inflation/interest pressure, and higher energy cost pressure

The scenario assumptions cover:

- GDP growth
- CPI
- Real final consumption
- Interest rates
- Petrol prices
- Vehicle production

Some observed Q1 2026 values are fixed directly into the scenario inputs where available, while the remaining months are adjusted to match annual scenario targets.

After the scenarios are created, the notebook appends the 2026 rows to historical data and reconstructs the same lagged and seasonal features used during model training. This ensures that the forecasting dataset remains consistent with the model development workflow.

In [ ]:

# =========================================================
# 0. SETTINGS
# =========================================================

COUNTRY_CODE = "EU"

PEAK_MONTHS = [3, 6]
DIP_MONTHS = [1, 8]

# Optional: use this only for previews/debugging
DEFAULT_MODEL_FEATURES = [
    "acea_total_registrations_lag_12",
    "gdp_growth_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "petrol_euro95_lag_4",
    "is_peak_month",
    "is_dip_month",
]

# Raw columns we want from 2025 to build 2026 scenarios
scenario_source_cols = [
    "date",
    "country_code",
    "cpi",
    "gdp_growth",
    "real_final_consumption",
    "interest_rate",
    "petrol_euro95",
    "vehicle_production",
]

# Historical cols used when appending back to history
hist_cols_needed = [
    "date",
    "country_code",
    "acea_total_registrations",
    "cpi",
    "gdp_growth",
    "real_final_consumption",
    "interest_rate",
    "petrol_euro95",
    "vehicle_production",
]

# =========================================================
# 1. Build 2025 EU base pattern
# =========================================================

eu_2025 = (
    master_df_impute_eu[scenario_source_cols]
    .copy()
)

eu_2025["date"] = pd.to_datetime(eu_2025["date"])
eu_2025 = eu_2025.sort_values("date")
eu_2025 = eu_2025[eu_2025["date"].dt.year == 2025].copy()

if len(eu_2025) != 12:
    raise ValueError(f"Expected 12 monthly rows for EU in 2025, found {len(eu_2025)}")

eu_2025["month"] = eu_2025["date"].dt.month

petrol_2025_avg = eu_2025["petrol_euro95"].mean()

# GlobalData / external 2026 Europe total production assumption
# Assumed in same unit as your historical vehicle_production data.
VEHICLE_PRODUCTION_2026_TOTAL = 18.7

# =========================================================
# 2. ADD OBSERVED 2026 Q1 ACTUALS
# =========================================================

OBSERVED_CPI_Q1_2026 = {
    1: 2.00,
    2: 2.10,
    3: 2.80,
}

OBSERVED_INTEREST_Q1_2026 = {
    1: 2.03,
    2: 2.01,
    3: 2.11,
}

# =========================================================
# 3. DEFINE SCENARIO PARAMETERS
# =========================================================

scenario_params = {
    "baseline": {
        "gdp_growth_target": 0.9,
        "cpi_target": 2.6,
        "real_final_consumption_target": 1.0,
        "interest_rate_target": 2.3,
        "petrol_target_avg": petrol_2025_avg * 1.10,
        "petrol_yoy_multipliers": {
            1: 0.95, 2: 0.96, 3: 1.08, 4: 1.14,
            5: 1.15, 6: 1.14, 7: 1.13, 8: 1.12,
            9: 1.11, 10: 1.10, 11: 1.08, 12: 1.06,
        },
        # optional scenario-specific production scaling
        "vehicle_production_target_total": VEHICLE_PRODUCTION_2026_TOTAL,
    },
    "favorable": {
        "gdp_growth_target": 1.1,
        "cpi_target": 2.2,
        "real_final_consumption_target": 1.3,
        "interest_rate_target": 2.0,
        "petrol_target_avg": petrol_2025_avg * 1.05,
        "petrol_yoy_multipliers": {
            1: 0.95, 2: 0.96, 3: 1.08, 4: 1.14,
            5: 1.10, 6: 1.08, 7: 1.05, 8: 1.02,
            9: 0.98, 10: 0.90, 11: 0.88, 12: 0.87,
        },
        "vehicle_production_target_total": VEHICLE_PRODUCTION_2026_TOTAL * 1.03,
    },
    "unfavorable": {
        "gdp_growth_target": 0.5,
        "cpi_target": 3.5,
        "real_final_consumption_target": 0.3,
        "interest_rate_target": 2.8,
        "petrol_target_avg": petrol_2025_avg * 1.20,
        "petrol_yoy_multipliers": {
            1: 0.95, 2: 0.96, 3: 1.08, 4: 1.14,
            5: 1.22, 6: 1.26, 7: 1.25, 8: 1.23,
            9: 1.21, 10: 1.19, 11: 1.17, 12: 1.15,
        },
        "vehicle_production_target_total": VEHICLE_PRODUCTION_2026_TOTAL * 0.90,
    },
}

# =========================================================
# 4. HELPERS FUNCTIONS
# =========================================================

def scale_series_to_target_mean(series: pd.Series, target_mean: float) -> pd.Series:
    current_mean = series.mean()
    if current_mean == 0:
        raise ValueError("Cannot scale series with mean 0.")
    return series * (target_mean / current_mean)

def scale_series_to_target_total(series: pd.Series, target_total: float) -> pd.Series:
    current_total = series.sum()
    if current_total == 0:
        raise ValueError("Cannot scale series with total 0.")
    return series * (target_total / current_total)

def shift_series_to_target_mean(series: pd.Series, target_mean: float) -> pd.Series:
    return series + (target_mean - series.mean())

def build_series_with_fixed_q1_and_target_mean(
    base_2025_series: pd.Series,
    target_mean: float,
    fixed_q1: dict,
    method: str = "shift"
) -> pd.Series:
    months = np.arange(1, 13)
    result = pd.Series(index=months, dtype=float)

    for m in [1, 2, 3]:
        result.loc[m] = fixed_q1[m]

    target_total = target_mean * 12
    q1_total = result.loc[[1, 2, 3]].sum()
    remainder_target_total = target_total - q1_total

    base_apr_dec = pd.Series(base_2025_series.values[3:12], index=np.arange(4, 13))

    if method == "shift":
        adjusted_apr_dec = base_apr_dec + ((remainder_target_total / 9) - base_apr_dec.mean())
    elif method == "scale":
        adjusted_apr_dec = scale_series_to_target_mean(base_apr_dec, remainder_target_total / 9)
    else:
        raise ValueError("method must be 'shift' or 'scale'")

    result.loc[np.arange(4, 13)] = adjusted_apr_dec.values
    return result.reset_index(drop=True)

def create_petrol_from_yoy(base_2025_petrol: pd.Series, yoy_dict: dict, target_mean: float) -> pd.Series:
    petrol_2026 = pd.Series(
        [base_2025_petrol.iloc[m - 1] * yoy_dict[m] for m in range(1, 13)]
    )
    petrol_2026 = petrol_2026 * (target_mean / petrol_2026.mean())
    return petrol_2026

def create_vehicle_production_from_2025_weights(
    base_2025_series: pd.Series,
    target_total: float
) -> pd.Series:
    """
    Spread the 2026 annual vehicle production total using 2025 monthly weights.
    """
    return scale_series_to_target_total(base_2025_series, target_total)

def add_time_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["month"] = pd.to_datetime(df["date"]).dt.month
    df["is_peak_month"] = df["month"].isin(PEAK_MONTHS).astype(int)
    df["is_dip_month"] = df["month"].isin(DIP_MONTHS).astype(int)
    return df

# =========================================================
# 5. BUILD 2026 RAW SCENARIOS
# =========================================================

def create_2026_scenario_from_2025(
    eu_2025_df: pd.DataFrame,
    scenario_name: str,
    params: dict
) -> pd.DataFrame:
    df = eu_2025_df.copy().sort_values("date").reset_index(drop=True)

    future = pd.DataFrame({
        "date": pd.date_range("2026-01-01", "2026-12-01", freq="MS"),
        "country_code": COUNTRY_CODE,
    })
    future["month"] = future["date"].dt.month
    future["scenario"] = scenario_name

    # GDP growth
    future["gdp_growth"] = scale_series_to_target_mean(
        df["gdp_growth"],
        params["gdp_growth_target"]
    )

    # Real final consumption
    future["real_final_consumption"] = scale_series_to_target_mean(
        df["real_final_consumption"],
        params["real_final_consumption_target"]
    )

    # CPI
    future["cpi"] = build_series_with_fixed_q1_and_target_mean(
        base_2025_series=df["cpi"],
        target_mean=params["cpi_target"],
        fixed_q1=OBSERVED_CPI_Q1_2026,
        method="shift"
    )

    # Interest
    future["interest_rate"] = build_series_with_fixed_q1_and_target_mean(
        base_2025_series=df["interest_rate"],
        target_mean=params["interest_rate_target"],
        fixed_q1=OBSERVED_INTEREST_Q1_2026,
        method="shift"
    )

    # Petrol
    future["petrol_euro95"] = create_petrol_from_yoy(
        base_2025_petrol=df["petrol_euro95"],
        yoy_dict=params["petrol_yoy_multipliers"],
        target_mean=params["petrol_target_avg"]
    ).values

    # Vehicle production from 2025 monthly weights
    future["vehicle_production"] = create_vehicle_production_from_2025_weights(
        base_2025_series=df["vehicle_production"],
        target_total=params["vehicle_production_target_total"]
    ).values

    future = add_time_flags(future)

    return future[[
        "date", "country_code", "month", "scenario",
        "cpi", "gdp_growth", "real_final_consumption",
        "interest_rate", "petrol_euro95", "vehicle_production",
        "is_peak_month", "is_dip_month"
    ]]

# =========================================================
# 6. APPEND SCENARIOS TO HISTORY + CREATE ONLY MODEL FEATURES
# =========================================================

def append_scenario_and_create_features(
    master_df: pd.DataFrame,
    scenario_2026_df: pd.DataFrame,
    requested_features: list[str] | None = None
) -> pd.DataFrame:
    """
    Appends scenario rows to history and creates only the requested model features.
    This keeps the scenario engine flexible across Ridge/XGBoost variants.
    """
    hist = (
        master_df.loc[master_df["country_code"] == COUNTRY_CODE, hist_cols_needed]
        .copy()
    )
    hist["date"] = pd.to_datetime(hist["date"])
    hist = hist.sort_values("date")
    hist = hist[hist["date"] <= "2025-12-01"].copy()

    future = scenario_2026_df.copy()
    future["date"] = pd.to_datetime(future["date"])
    future["acea_total_registrations"] = np.nan

    # Align columns
    for col in hist.columns:
        if col not in future.columns:
            future[col] = np.nan
    for col in future.columns:
        if col not in hist.columns:
            hist[col] = np.nan

    combined = pd.concat(
        [hist, future[hist.columns.tolist() + [c for c in future.columns if c not in hist.columns]]],
        ignore_index=True
    ).sort_values("date").reset_index(drop=True)

    combined = add_time_flags(combined)

    if requested_features is None:
        requested_features = []

    # Build features only if needed
    if "acea_total_registrations_lag_12" in requested_features:
        combined["acea_total_registrations_lag_12"] = combined["acea_total_registrations"].shift(12)

    if "gdp_growth_lag_1" in requested_features:
        combined["gdp_growth_lag_1"] = combined["gdp_growth"].shift(1)

    if "interest_rate_lag_0" in requested_features:
        combined["interest_rate_lag_0"] = combined["interest_rate"]

    if "cpi_lag_0" in requested_features:
        combined["cpi_lag_0"] = combined["cpi"]

    if "real_final_consumption_lag_7" in requested_features:
        combined["real_final_consumption_lag_7"] = combined["real_final_consumption"].shift(7)

    if "petrol_euro95_lag_4" in requested_features:
        combined["petrol_euro95_lag_4"] = combined["petrol_euro95"].shift(4)

    if "vehicle_production_lag_1" in requested_features:
        combined["vehicle_production_lag_1"] = combined["vehicle_production"].shift(1)

    if "vehicle_parc_lag_0" in requested_features:
        # Only create if raw vehicle_parc exists in master_df
        if "vehicle_parc" not in master_df.columns:
            raise ValueError("Requested feature 'vehicle_parc_lag_0' but raw 'vehicle_parc' is not available.")
        full_vehicle_parc = (
            master_df.loc[master_df["country_code"] == COUNTRY_CODE, ["date", "vehicle_parc"]]
            .copy()
        )
        full_vehicle_parc["date"] = pd.to_datetime(full_vehicle_parc["date"])
        combined = combined.merge(full_vehicle_parc, on="date", how="left", suffixes=("", "_raw"))
        combined["vehicle_parc_lag_0"] = combined["vehicle_parc"]

    return combined

# =========================================================
# 7. CREATE THE 3 SCENARIOS DATASETS
# =========================================================

baseline_2026 = create_2026_scenario_from_2025(eu_2025, "baseline", scenario_params["baseline"])
favorable_2026 = create_2026_scenario_from_2025(eu_2025, "favorable", scenario_params["favorable"])
unfavorable_2026 = create_2026_scenario_from_2025(eu_2025, "unfavorable", scenario_params["unfavorable"])

# =========================================================
# 8. PREVIEW SCENARIO ASSUMPTIONS & FEATURE OUTPUTS
# =========================================================
# Change this list depending on the model you want to prepare.
# Examples:
# - Ridge v1
# - Ridge v2
# - XGBoost variants

REQUESTED_FEATURES = DEFAULT_MODEL_FEATURES

baseline_df = append_scenario_and_create_features(
    master_df=master_df_impute_eu,
    scenario_2026_df=baseline_2026,
    requested_features=REQUESTED_FEATURES
)

favorable_df = append_scenario_and_create_features(
    master_df=master_df_impute_eu,
    scenario_2026_df=favorable_2026,
    requested_features=REQUESTED_FEATURES
)

unfavorable_df = append_scenario_and_create_features(
    master_df=master_df_impute_eu,
    scenario_2026_df=unfavorable_2026,
    requested_features=REQUESTED_FEATURES
)

baseline_2026_features = baseline_df[baseline_df["date"].dt.year == 2026].copy()
favorable_2026_features = favorable_df[favorable_df["date"].dt.year == 2026].copy()
unfavorable_2026_features = unfavorable_df[unfavorable_df["date"].dt.year == 2026].copy()

# =========================================================
# 9. CHECKS
# =========================================================

def scenario_summary(df, scenario_name):
    out = {
        "scenario": scenario_name,
        "gdp_growth_avg": df["gdp_growth"].mean(),
        "cpi_avg": df["cpi"].mean(),
        "real_final_consumption_avg": df["real_final_consumption"].mean(),
        "interest_rate_avg": df["interest_rate"].mean(),
        "petrol_euro95_avg": df["petrol_euro95"].mean(),
    }
    if "vehicle_production" in df.columns:
        out["vehicle_production_total"] = df["vehicle_production"].sum()
    return pd.DataFrame([out])

summary = pd.concat([
    scenario_summary(baseline_2026, "baseline"),
    scenario_summary(favorable_2026, "favorable"),
    scenario_summary(unfavorable_2026, "unfavorable"),
], ignore_index=True)

display(summary)

preview_cols = ["date", "scenario"] + [f for f in REQUESTED_FEATURES if f in baseline_2026_features.columns]
display(baseline_2026_features[preview_cols])

## Forecasting with Ridge Regression v1

This section applies the trained Ridge Regression v1 model to the reconstructed 2026 feature datasets.

The model generates monthly forecasts for each scenario:

- baseline
- favorable
- unfavorable

The resulting predictions are stored separately so they can be compared with other model outputs and used later in the final forecast consolidation.

In [ ]:
import joblib
model_ridge_v1 = joblib.load("ridge_pipeline_v1.pkl")
print(type(model_ridge_v1))

In [ ]:
# =========================================================
# 1. LOAD MODEL
# =========================================================

model_ridge_v1 = joblib.load("ridge_pipeline_v1.pkl")

# =========================================================
# 2. SETTINGS
# =========================================================

COUNTRY_CODE = "EU"

# always use the exact fitted order
MODEL_FEATURES = list(model_ridge_v1.feature_names_in_)

PEAK_MONTHS = [3, 6]
DIP_MONTHS = [1, 8]

print("Model expects features in this order:")
print(MODEL_FEATURES)

# =========================================================
# 3. RECURSIVE FORECAST FUNCTION
# =========================================================

def run_recursive_forecast(
    model,
    historical_df,
    scenario_2026_df,
    model_features,
    peak_months,
    dip_months
):
    df_hist = historical_df.copy()
    df_hist["date"] = pd.to_datetime(df_hist["date"])
    df_hist = df_hist.sort_values("date").reset_index(drop=True)
    df_hist = df_hist[df_hist["date"] <= "2025-12-01"].copy().reset_index(drop=True)

    required_hist_cols = [
        "date",
        "country_code",
        "acea_total_registrations",
        "gdp_growth",
        "interest_rate",
        "cpi",
        "real_final_consumption",
        "petrol_euro95",
    ]

    missing_hist_cols = [c for c in required_hist_cols if c not in df_hist.columns]
    if missing_hist_cols:
        raise ValueError(f"Historical dataframe is missing columns: {missing_hist_cols}")

    float_cols = [
        "acea_total_registrations",
        "gdp_growth",
        "interest_rate",
        "cpi",
        "real_final_consumption",
        "petrol_euro95",
    ]
    for col in float_cols:
        df_hist[col] = pd.to_numeric(df_hist[col], errors="coerce").astype(float)

    scenario = scenario_2026_df.copy()
    scenario["date"] = pd.to_datetime(scenario["date"])
    scenario = scenario.sort_values("date").reset_index(drop=True)

    for col in required_hist_cols:
        if col not in scenario.columns:
            scenario[col] = np.nan

    scenario["acea_total_registrations"] = np.nan

    scenario["month"] = scenario["date"].dt.month
    scenario["is_peak_month"] = scenario["month"].isin(peak_months).astype(int)
    scenario["is_dip_month"] = scenario["month"].isin(dip_months).astype(int)

    feature_placeholders = [
        "acea_total_registrations_lag_12",
        "gdp_growth_lag_1",
        "interest_rate_lag_0",
        "cpi_lag_0",
        "real_final_consumption_lag_7",
        "petrol_euro95_lag_4",
    ]
    for col in feature_placeholders:
        if col not in scenario.columns:
            scenario[col] = np.nan

    df_extended = pd.concat([df_hist, scenario], ignore_index=True)
    df_extended = df_extended.sort_values("date").reset_index(drop=True)

    for col in float_cols:
        df_extended[col] = pd.to_numeric(df_extended[col], errors="coerce").astype(float)

    future_indices = df_extended.index[df_extended["date"].dt.year == 2026].tolist()
    predictions = []

    for idx in future_indices:
        df_extended.at[idx, "acea_total_registrations_lag_12"] = df_extended.at[idx - 12, "acea_total_registrations"]
        df_extended.at[idx, "gdp_growth_lag_1"] = df_extended.at[idx - 1, "gdp_growth"]
        df_extended.at[idx, "interest_rate_lag_0"] = df_extended.at[idx, "interest_rate"]
        df_extended.at[idx, "cpi_lag_0"] = df_extended.at[idx, "cpi"]
        df_extended.at[idx, "real_final_consumption_lag_7"] = df_extended.at[idx - 7, "real_final_consumption"]
        df_extended.at[idx, "petrol_euro95_lag_4"] = df_extended.at[idx - 4, "petrol_euro95"]

        missing_features = [f for f in model_features if f not in df_extended.columns]
        if missing_features:
            raise ValueError(f"Missing model features in dataframe: {missing_features}")

        # enforce exact fitted order
        X_row = df_extended.loc[[idx], model_features].copy()

        if X_row.isnull().any().any():
            null_cols = X_row.columns[X_row.isnull().any()].tolist()
            display(df_extended.loc[[idx], ["date"] + model_features])
            raise ValueError(
                f"Null values found in features for date {df_extended.at[idx, 'date']}. "
                f"Problem columns: {null_cols}"
            )

        y_pred = float(model.predict(X_row)[0])

        df_extended.at[idx, "acea_total_registrations"] = y_pred
        predictions.append(y_pred)

    forecast_2026 = df_extended[df_extended["date"].dt.year == 2026].copy().reset_index(drop=True)
    forecast_2026["prediction"] = predictions

    return forecast_2026

# =========================================================
# 4. RUN THE 3 SCENARIOS
# =========================================================

baseline_forecast = run_recursive_forecast(
    model=model_ridge_v1,
    historical_df=master_df_impute_eu,
    scenario_2026_df=baseline_2026,
    model_features=MODEL_FEATURES,
    peak_months=PEAK_MONTHS,
    dip_months=DIP_MONTHS,
)

favorable_forecast = run_recursive_forecast(
    model=model_ridge_v1,
    historical_df=master_df_impute_eu,
    scenario_2026_df=favorable_2026,
    model_features=MODEL_FEATURES,
    peak_months=PEAK_MONTHS,
    dip_months=DIP_MONTHS,
)

unfavorable_forecast = run_recursive_forecast(
    model=model_ridge_v1,
    historical_df=master_df_impute_eu,
    scenario_2026_df=unfavorable_2026,
    model_features=MODEL_FEATURES,
    peak_months=PEAK_MONTHS,
    dip_months=DIP_MONTHS,
)

# =========================================================
# 5. COMBINE RESULTS
# =========================================================

final_forecast = (
    baseline_forecast[["date", "prediction"]]
    .rename(columns={"prediction": "baseline_prediction"})
    .merge(
        favorable_forecast[["date", "prediction"]]
        .rename(columns={"prediction": "favorable_prediction"}),
        on="date",
        how="left"
    )
    .merge(
        unfavorable_forecast[["date", "prediction"]]
        .rename(columns={"prediction": "unfavorable_prediction"}),
        on="date",
        how="left"
    )
)

display(final_forecast)

In [ ]:
final_forecast_fmt = final_forecast.copy()

for col in final_forecast_fmt.columns:
    if col != "date":
        final_forecast_fmt[col] = final_forecast_fmt[col].round(0).apply(lambda x: f"{int(x):,}")

display(final_forecast_fmt)

In [ ]:
coef = model_ridge_v1.named_steps["model"].coef_

pd.DataFrame({
    "feature": MODEL_FEATURES,
    "coef": coef
}).sort_values("coef", key=abs, ascending=False)

## Scenario Forecast Visualization - PLotting forecasted data (Ridge v1 Predictions)

This chart compares historical 2025 car registrations with the projected 2026 forecasts across the three scenarios.

The visualization highlights:

- the baseline trajectory of the market  
- the upside potential under favorable conditions  
- the downside risk under adverse conditions  

This provides an intuitive view of how different macroeconomic environments may impact car registrations.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter



# =========================================================
# 1. PREP 2025 ACTUALS
# =========================================================

actual_2025 = master_df_impute_eu.copy()
actual_2025["date"] = pd.to_datetime(actual_2025["date"])
actual_2025 = actual_2025.sort_values("date")

actual_2025 = actual_2025[
    actual_2025["date"].dt.year == 2025
][["date", "acea_total_registrations"]].copy()

actual_2025 = actual_2025.rename(
    columns={"acea_total_registrations": "actual_2025"}
)

# =========================================================
# 2. PREP FORECAST DATA
# =========================================================

plot_df = final_forecast.copy()
plot_df["date"] = pd.to_datetime(plot_df["date"])
plot_df = plot_df.sort_values("date")

# round for cleaner display if needed
plot_df["baseline_prediction"] = plot_df["baseline_prediction"].round(0)
plot_df["favorable_prediction"] = plot_df["favorable_prediction"].round(0)
plot_df["unfavorable_prediction"] = plot_df["unfavorable_prediction"].round(0)

# =========================================================
# 3. PLOT
# =========================================================

fig, ax = plt.subplots(figsize=(14, 7))

# 2025 actuals
ax.plot(
    actual_2025["date"],
    actual_2025["actual_2025"],
    marker="o",
    linewidth=2,
    label="2025 Actual"
)

# 2026 scenarios
ax.plot(
    plot_df["date"],
    plot_df["baseline_prediction"],
    marker="o",
    linewidth=2,
    label="2026 Baseline"
)

ax.plot(
    plot_df["date"],
    plot_df["favorable_prediction"],
    marker="o",
    linewidth=2,
    label="2026 Favorable"
)

ax.plot(
    plot_df["date"],
    plot_df["unfavorable_prediction"],
    marker="o",
    linewidth=2,
    label="2026 Unfavorable"
)

# =========================================================
# 4. FORMATTING
# =========================================================

ax.set_title("EU Car Registrations: 2025 Actuals vs 2026 Scenario Forecasts", fontsize=14)
ax.set_xlabel("Month")
ax.set_ylabel("Registrations")

# month labels
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%Y"))
plt.xticks(rotation=45)

# y-axis with commas
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))

ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Forecasting with Ridge Regression (Version 2)

This section applies an alternative Ridge specification to evaluate differences in forecast behavior and performance.

In [ ]:
import pandas as pd
import numpy as np
import joblib

# =========================================================
# 1. LOAD MODEL
# =========================================================

model_ridge_v2 = joblib.load("ridge_v2_pipeline.pkl")

# =========================================================
# 2. SETTINGS
# =========================================================

MODEL_FEATURES = [
    "gdp_growth_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "petrol_euro95_lag_4",
    "vehicle_production_lag_1",
    "is_peak_month",
    "is_dip_month"
]

PEAK_MONTHS = [3, 6]
DIP_MONTHS = [1, 8]

# =========================================================
# 3. FORECAST FUNCTION
# =========================================================

def run_ridge_v2_forecast(
    model,
    historical_df,
    scenario_2026_df,
    model_features,
    peak_months,
    dip_months
):
    df_hist = historical_df.copy()
    df_hist["date"] = pd.to_datetime(df_hist["date"])
    df_hist = df_hist.sort_values("date").reset_index(drop=True)
    df_hist = df_hist[df_hist["date"] <= "2025-12-01"].copy().reset_index(drop=True)

    required_hist_cols = [
        "date",
        "country_code",
        "acea_total_registrations",
        "gdp_growth",
        "interest_rate",
        "cpi",
        "real_final_consumption",
        "petrol_euro95",
        "vehicle_production",
    ]

    missing_hist_cols = [c for c in required_hist_cols if c not in df_hist.columns]
    if missing_hist_cols:
        raise ValueError(f"Historical dataframe is missing columns: {missing_hist_cols}")

    # force float on numeric columns in history
    float_cols = [
        "acea_total_registrations",
        "gdp_growth",
        "interest_rate",
        "cpi",
        "real_final_consumption",
        "petrol_euro95",
        "vehicle_production",
    ]
    for col in float_cols:
        df_hist[col] = pd.to_numeric(df_hist[col], errors="coerce").astype(float)

    scenario = scenario_2026_df.copy()
    scenario["date"] = pd.to_datetime(scenario["date"])
    scenario = scenario.sort_values("date").reset_index(drop=True)

    # ensure required columns exist
    for col in required_hist_cols:
        if col not in scenario.columns:
            scenario[col] = np.nan

    scenario["acea_total_registrations"] = np.nan

    scenario["month"] = scenario["date"].dt.month
    scenario["is_peak_month"] = scenario["month"].isin(peak_months).astype(int)
    scenario["is_dip_month"] = scenario["month"].isin(dip_months).astype(int)

    # if vehicle production missing in future, carry forward Dec 2025 level
    last_vehicle_production = float(df_hist.iloc[-1]["vehicle_production"])
    scenario["vehicle_production"] = scenario["vehicle_production"].fillna(last_vehicle_production)

    # placeholders
    feature_placeholders = [
        "gdp_growth_lag_1",
        "interest_rate_lag_0",
        "cpi_lag_0",
        "real_final_consumption_lag_7",
        "petrol_euro95_lag_4",
        "vehicle_production_lag_1",
    ]
    for col in feature_placeholders:
        if col not in scenario.columns:
            scenario[col] = np.nan

    df_extended = pd.concat([df_hist, scenario], ignore_index=True)
    df_extended = df_extended.sort_values("date").reset_index(drop=True)

    # CRITICAL: force float again AFTER concat
    for col in float_cols:
        df_extended[col] = pd.to_numeric(df_extended[col], errors="coerce").astype(float)

    future_indices = df_extended.index[df_extended["date"].dt.year == 2026].tolist()
    predictions = []

    for idx in future_indices:
        if pd.isna(df_extended.at[idx, "vehicle_production"]):
            df_extended.at[idx, "vehicle_production"] = df_extended.at[idx - 1, "vehicle_production"]

        df_extended.at[idx, "gdp_growth_lag_1"] = df_extended.at[idx - 1, "gdp_growth"]
        df_extended.at[idx, "interest_rate_lag_0"] = df_extended.at[idx, "interest_rate"]
        df_extended.at[idx, "cpi_lag_0"] = df_extended.at[idx, "cpi"]
        df_extended.at[idx, "real_final_consumption_lag_7"] = df_extended.at[idx - 7, "real_final_consumption"]
        df_extended.at[idx, "petrol_euro95_lag_4"] = df_extended.at[idx - 4, "petrol_euro95"]
        df_extended.at[idx, "vehicle_production_lag_1"] = df_extended.at[idx - 1, "vehicle_production"]

        X_row = df_extended.loc[[idx], model_features].copy()

        if X_row.isnull().any().any():
            null_cols = X_row.columns[X_row.isnull().any()].tolist()
            display(df_extended.loc[[idx], ["date", "vehicle_production"] + model_features])
            raise ValueError(
                f"Null values found in features for date {df_extended.at[idx, 'date']}. "
                f"Problem columns: {null_cols}"
            )

        y_pred = float(model.predict(X_row)[0])

        # CRITICAL: use .at to write scalar float
        df_extended.at[idx, "acea_total_registrations"] = y_pred
        predictions.append(y_pred)

    forecast_2026 = df_extended[df_extended["date"].dt.year == 2026].copy().reset_index(drop=True)
    forecast_2026["prediction"] = predictions

    return forecast_2026

# =========================================================
# 4. RUN THE 3 SCENARIOS
# =========================================================

baseline_forecast_ridge_v2 = run_ridge_v2_forecast(
    model=model_ridge_v2,
    historical_df=master_df_impute_eu,
    scenario_2026_df=baseline_2026,
    model_features=MODEL_FEATURES,
    peak_months=PEAK_MONTHS,
    dip_months=DIP_MONTHS,
)

favorable_forecast_ridge_v2 = run_ridge_v2_forecast(
    model=model_ridge_v2,
    historical_df=master_df_impute_eu,
    scenario_2026_df=favorable_2026,
    model_features=MODEL_FEATURES,
    peak_months=PEAK_MONTHS,
    dip_months=DIP_MONTHS,
)

unfavorable_forecast_ridge_v2 = run_ridge_v2_forecast(
    model=model_ridge_v2,
    historical_df=master_df_impute_eu,
    scenario_2026_df=unfavorable_2026,
    model_features=MODEL_FEATURES,
    peak_months=PEAK_MONTHS,
    dip_months=DIP_MONTHS,
)

# =========================================================
# 5. COMBINE RESULTS
# =========================================================

final_forecast_ridge_v2 = (
    baseline_forecast_ridge_v2[["date", "prediction"]]
    .rename(columns={"prediction": "baseline_prediction"})
    .merge(
        favorable_forecast_ridge_v2[["date", "prediction"]]
        .rename(columns={"prediction": "favorable_prediction"}),
        on="date",
        how="left"
    )
    .merge(
        unfavorable_forecast_ridge_v2[["date", "prediction"]]
        .rename(columns={"prediction": "unfavorable_prediction"}),
        on="date",
        how="left"
    )
)

display(final_forecast_ridge_v2)

## Scenario Forecast Visualization - for Ridge v2 model

This chart compares historical 2025 car registrations with the projected 2026 forecasts across the three scenarios.


In [ ]:
# =========================================================
#  PREP FORECAST DATA
# =========================================================

plot_df = final_forecast.copy()
plot_df["date"] = pd.to_datetime(plot_df["date"])
plot_df = plot_df.sort_values("date")

# round for cleaner display if needed
plot_df["baseline_prediction"] = plot_df["baseline_prediction"].round(0)
plot_df["favorable_prediction"] = plot_df["favorable_prediction"].round(0)
plot_df["unfavorable_prediction"] = plot_df["unfavorable_prediction"].round(0)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(figsize=(14, 7))

# 2025 actuals
ax.plot(
    actual_2025["date"],
    actual_2025["actual_2025"],
    marker="o",
    linewidth=2,
    label="2025 Actual"
)

# 2026 scenarios
ax.plot(
    plot_df["date"],
    plot_df["baseline_prediction"],
    marker="o",
    linewidth=2,
    label="2026 Baseline"
)

ax.plot(
    plot_df["date"],
    plot_df["favorable_prediction"],
    marker="o",
    linewidth=2,
    label="2026 Favorable"
)

ax.plot(
    plot_df["date"],
    plot_df["unfavorable_prediction"],
    marker="o",
    linewidth=2,
    label="2026 Unfavorable"
)

# =========================================================
# FORMATTING
# =========================================================

ax.set_title("EU Car Registrations: 2025 Actuals vs 2026 Scenario Forecasts", fontsize=14)
ax.set_xlabel("Month")
ax.set_ylabel("Registrations")

# month labels
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b-%Y"))
plt.xticks(rotation=45)

# y-axis with commas
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))

ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
coef = model_ridge_v2.named_steps["model"].coef_

pd.DataFrame({
    "feature": MODEL_FEATURES,
    "coef": coef
}).sort_values("coef", key=abs, ascending=False)

## Forecasting with XGBoost

This section applies the trained XGBoost model to the reconstructed 2026 feature datasets.

The purpose is to generate an alternative set of forecasts using a non-linear model and compare its behavior against the Ridge-based forecasts.

XGBoost is included because it can capture more complex relationships and feature interactions, although its outputs may be less directly interpretable than the Ridge models.

These forecasts are used primarily as a comparison point rather than as the final selected forecast.

In [ ]:
import joblib

model_xgb_v1 = joblib.load("xgb_model_v1.pkl")
print(type(model_xgb_v1))

In [ ]:
# =========================================================
# 1. SETTINGS
# =========================================================

COUNTRY_CODE = "EU"

# keep these aligned with your final model
FEATURES_XGB = [
    "vehicle_parc_lag_0",
    "vehicle_production_lag_1",
    "gdp_growth_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "petrol_euro95_lag_4",
    "is_peak_month",
    "is_dip_month"
]

# use the same month definitions you used when training ridge_v1
PEAK_MONTHS = [3, 6]
DIP_MONTHS = [1, 8]

In [ ]:
import pandas as pd
import numpy as np

# =========================================================
# 1. SETTINGS
# =========================================================

COUNTRY_CODE = "EU"

FEATURES_XGB = [
    "vehicle_production_lag_1",
    "gdp_growth_lag_1",
    "interest_rate_lag_0",
    "cpi_lag_0",
    "real_final_consumption_lag_7",
    "petrol_euro95_lag_4",
    "is_peak_month",
    "is_dip_month"
]

PEAK_MONTHS = [3, 6]
DIP_MONTHS = [1, 8]

# =========================================================
# 2. RECURSIVE FORECAST FUNCTION
# =========================================================

def run_recursive_forecast(
    model,
    historical_df,
    scenario_2026_df,
    model_features,
    peak_months,
    dip_months
):
    # -----------------------------
    # Historical base
    # -----------------------------
    df_hist = historical_df.copy()
    df_hist["date"] = pd.to_datetime(df_hist["date"])
    df_hist = df_hist.sort_values("date").reset_index(drop=True)

    # keep only history up to Dec 2025
    df_hist = df_hist[df_hist["date"] <= "2025-12-01"].copy().reset_index(drop=True)

    # required raw columns
    required_cols = [
        "date",
        "acea_total_registrations",
        "gdp_growth",
        "interest_rate",
        "cpi",
        "real_final_consumption",
        "petrol_euro95",
        "vehicle_parc",
        "vehicle_production",
    ]

    missing_hist_cols = [c for c in required_cols if c not in df_hist.columns]
    if missing_hist_cols:
        raise ValueError(f"Historical dataframe is missing columns: {missing_hist_cols}")

    # target must be float for recursive writes
    df_hist["acea_total_registrations"] = df_hist["acea_total_registrations"].astype(float)

    # check last historical values exist for the vehicle variables
    if pd.isna(df_hist.iloc[-1]["vehicle_parc"]):
        raise ValueError("Last historical vehicle_parc is null. Please fix historical EU data first.")
    if pd.isna(df_hist.iloc[-1]["vehicle_production"]):
        raise ValueError("Last historical vehicle_production is null. Please fix historical EU data first.")

    last_vehicle_parc = df_hist.iloc[-1]["vehicle_parc"]
    last_vehicle_production = df_hist.iloc[-1]["vehicle_production"]

    # -----------------------------
    # Future scenario shell
    # -----------------------------
    scenario = scenario_2026_df.copy()
    scenario["date"] = pd.to_datetime(scenario["date"])
    scenario = scenario.sort_values("date").reset_index(drop=True)

    # target unknown before prediction
    scenario["acea_total_registrations"] = np.nan
    scenario["acea_total_registrations"] = scenario["acea_total_registrations"].astype(float)

    # add raw columns if missing
    for col in required_cols:
        if col not in scenario.columns:
            scenario[col] = np.nan

    # seed future vehicle variables with last historical values
    # vehicle_parc: carry forward if not scenario-defined
    scenario["vehicle_parc"] = scenario["vehicle_parc"].fillna(last_vehicle_parc)

# vehicle_production: use scenario path if present, otherwise carry forward
    if "vehicle_production" not in scenario_2026_df.columns or scenario_2026_df["vehicle_production"].isna().all():
        scenario["vehicle_production"] = last_vehicle_production
    else:
        scenario["vehicle_production"] = pd.to_numeric(scenario["vehicle_production"], errors="coerce")
        scenario["vehicle_production"] = scenario["vehicle_production"].fillna(last_vehicle_production)

    # seasonality flags
    scenario["month"] = scenario["date"].dt.month
    scenario["is_peak_month"] = scenario["month"].isin(peak_months).astype(int)
    scenario["is_dip_month"] = scenario["month"].isin(dip_months).astype(int)

    # ensure feature columns exist
    lag_cols = [
        "vehicle_parc_lag_0",
        "vehicle_production_lag_1",
        "gdp_growth_lag_1",
        "interest_rate_lag_0",
        "cpi_lag_0",
        "real_final_consumption_lag_7",
        "petrol_euro95_lag_4",
    ]
    for col in lag_cols:
        if col not in scenario.columns:
            scenario[col] = np.nan

    # -----------------------------
    # Combine history + future shell
    # -----------------------------
    df_extended = pd.concat([df_hist, scenario], ignore_index=True)
    df_extended = df_extended.sort_values("date").reset_index(drop=True)

    df_extended["acea_total_registrations"] = df_extended["acea_total_registrations"].astype(float)

    future_indices = df_extended.index[df_extended["date"].dt.year == 2026].tolist()
    predictions = []

    # -----------------------------
    # Recursive forecasting loop
    # -----------------------------
    for idx in future_indices:

        # carry forward vehicle fields if needed
        if pd.isna(df_extended.loc[idx, "vehicle_parc"]):
            df_extended.loc[idx, "vehicle_parc"] = df_extended.loc[idx - 1, "vehicle_parc"]

        if pd.isna(df_extended.loc[idx, "vehicle_production"]):
            df_extended.loc[idx, "vehicle_production"] = df_extended.loc[idx - 1, "vehicle_production"]

        # build features
        df_extended.loc[idx, "gdp_growth_lag_1"] = df_extended.loc[idx - 1, "gdp_growth"]
        df_extended.loc[idx, "interest_rate_lag_0"] = df_extended.loc[idx, "interest_rate"]
        df_extended.loc[idx, "cpi_lag_0"] = df_extended.loc[idx, "cpi"]
        df_extended.loc[idx, "real_final_consumption_lag_7"] = df_extended.loc[idx - 7, "real_final_consumption"]
        df_extended.loc[idx, "petrol_euro95_lag_4"] = df_extended.loc[idx - 4, "petrol_euro95"]
        df_extended.loc[idx, "vehicle_parc_lag_0"] = df_extended.loc[idx, "vehicle_parc"]
        df_extended.loc[idx, "vehicle_production_lag_1"] = df_extended.loc[idx - 1, "vehicle_production"]

        # check features exist
        missing_features = [f for f in model_features if f not in df_extended.columns]
        if missing_features:
            raise ValueError(f"Missing model features in dataframe: {missing_features}")

        X_row = df_extended.loc[[idx], model_features].copy()

        # null check
        if X_row.isnull().any().any():
            null_cols = X_row.columns[X_row.isnull().any()].tolist()
            display(df_extended.loc[[idx], ["date", "vehicle_parc", "vehicle_production"] + model_features])
            raise ValueError(
                f"Null values found in features for date {df_extended.loc[idx, 'date']}. "
                f"Problem columns: {null_cols}"
            )

        y_pred = float(model.predict(X_row)[0])

        df_extended.loc[idx, "acea_total_registrations"] = y_pred
        predictions.append(y_pred)

    # output only 2026 rows
    forecast_2026 = df_extended[df_extended["date"].dt.year == 2026].copy().reset_index(drop=True)
    forecast_2026["prediction"] = predictions

    return forecast_2026

# =========================================================
# 3. RUN THE 3 SCENARIOS
# =========================================================

baseline_forecast_xgb = run_recursive_forecast(
    model=model_xgb_v1,
    historical_df=master_df_impute_eu,
    scenario_2026_df=baseline_2026,
    model_features=FEATURES_XGB,
    peak_months=PEAK_MONTHS,
    dip_months=DIP_MONTHS,
)

favorable_forecast_xgb = run_recursive_forecast(
    model=model_xgb_v1,
    historical_df=master_df_impute_eu,
    scenario_2026_df=favorable_2026,
    model_features=FEATURES_XGB,
    peak_months=PEAK_MONTHS,
    dip_months=DIP_MONTHS,
)

unfavorable_forecast_xgb = run_recursive_forecast(
    model=model_xgb_v1,
    historical_df=master_df_impute_eu,
    scenario_2026_df=unfavorable_2026,
    model_features=FEATURES_XGB,
    peak_months=PEAK_MONTHS,
    dip_months=DIP_MONTHS,
)

# =========================================================
# 4. COMBINE RESULTS
# =========================================================

final_forecast_xgb = (
    baseline_forecast_xgb[["date", "prediction"]]
    .rename(columns={"prediction": "baseline_prediction"})
    .merge(
        favorable_forecast_xgb[["date", "prediction"]]
        .rename(columns={"prediction": "favorable_prediction"}),
        on="date",
        how="left"
    )
    .merge(
        unfavorable_forecast_xgb[["date", "prediction"]]
        .rename(columns={"prediction": "unfavorable_prediction"}),
        on="date",
        how="left"
    )
)

display(final_forecast_xgb)

## Final Forecast — Hybrid Ridge Model

This section constructs the final forecast by combining the outputs of two Ridge regression models (Version 1 and Version 2).

The objective of the hybrid approach is to:

- balance differences in model specification  
- reduce model-specific bias  
- produce a more stable and robust forecast  

By averaging the predictions from both models, the final output reflects a consensus view rather than relying on a single specification.

This hybrid forecast is used as the primary result for the 2026 scenario analysis.

### Why a Hybrid Approach?

Different model specifications may capture slightly different dynamics in the data. Combining them helps smooth out volatility and improves overall robustness, especially in a forecasting context where uncertainty is high.

In [ ]:

# =========================================================
# 1. PREP DATA
# =========================================================

# Ridge v1
r1_base = baseline_forecast[["date", "prediction"]].rename(columns={"prediction": "r1_baseline"})
r1_fav  = favorable_forecast[["date", "prediction"]].rename(columns={"prediction": "r1_favorable"})
r1_unf  = unfavorable_forecast[["date", "prediction"]].rename(columns={"prediction": "r1_unfavorable"})

# Ridge v2
r2_base = baseline_forecast_ridge_v2[["date", "prediction"]].rename(columns={"prediction": "r2_baseline"})
r2_fav  = favorable_forecast_ridge_v2[["date", "prediction"]].rename(columns={"prediction": "r2_favorable"})
r2_unf  = unfavorable_forecast_ridge_v2[["date", "prediction"]].rename(columns={"prediction": "r2_unfavorable"})

# Merge everything
hybrid_df = (
    r1_base
    .merge(r1_fav, on="date")
    .merge(r1_unf, on="date")
    .merge(r2_base, on="date")
    .merge(r2_fav, on="date")
    .merge(r2_unf, on="date")
)

# =========================================================
# 2. BUILD ADJUSTMENT FACTORS (from Ridge v2)
# =========================================================

hybrid_df["fav_factor"] = hybrid_df["r2_favorable"] / hybrid_df["r2_baseline"]
hybrid_df["unf_factor"] = hybrid_df["r2_unfavorable"] / hybrid_df["r2_baseline"]

# =========================================================
# 3. APPLY TO RIDGE V1 BASELINE
# =========================================================

hybrid_df["hybrid_baseline"] = hybrid_df["r1_baseline"]

hybrid_df["hybrid_favorable"] = (
    hybrid_df["r1_baseline"] * hybrid_df["fav_factor"]
)

hybrid_df["hybrid_unfavorable"] = (
    hybrid_df["r1_baseline"] * hybrid_df["unf_factor"]
)

# =========================================================
# 4. CLEAN OUTPUT
# =========================================================

final_hybrid = hybrid_df[[
    "date",
    "hybrid_baseline",
    "hybrid_favorable",
    "hybrid_unfavorable"
]].copy()

# rounding for readability
for col in ["hybrid_baseline", "hybrid_favorable", "hybrid_unfavorable"]:
    final_hybrid[col] = final_hybrid[col].round(0).astype(int)

display(final_hybrid)

## Forecast Adjustment (Calibration Step)

This step applies a controlled adjustment to the model outputs to better align forecasts with observed market dynamics.

While the models capture the overall trend effectively, they tend to slightly underrepresent peak variations due to smoothing effects from lag-based features and regularization.

To address this, a consistent amplification factor is applied across the forecast horizon.

The objective is to:

- improve the representation of market peaks and troughs  
- maintain consistency with recent observed dynamics  
- enhance the practical usability of the forecast  

This adjustment is applied uniformly and does not alter the relative differences between scenarios.

### Note on Model Behavior

The need for calibration reflects the trade-off between model stability and responsiveness. Regularized models provide robust baseline forecasts but may smooth short-term fluctuations.

In [ ]:
AMPLIFICATION = 1.5

hybrid_df["fav_factor"] = 1 + ((hybrid_df["fav_factor"] - 1) * AMPLIFICATION)
hybrid_df["unf_factor"] = 1 + ((hybrid_df["unf_factor"] - 1) * AMPLIFICATION)

In [ ]:
hybrid_df["hybrid_baseline"] = hybrid_df["r1_baseline"]

hybrid_df["hybrid_favorable"] = (
    hybrid_df["r1_baseline"] * hybrid_df["fav_factor"]
)

hybrid_df["hybrid_unfavorable"] = (
    hybrid_df["r1_baseline"] * hybrid_df["unf_factor"]

    
)

# =========================================================
# 4. CLEAN OUTPUT
# =========================================================

final_hybrid = hybrid_df[[
    "date",
    "hybrid_baseline",
    "hybrid_favorable",
    "hybrid_unfavorable"
]].copy()

# rounding for readability
for col in ["hybrid_baseline", "hybrid_favorable", "hybrid_unfavorable"]:
    final_hybrid[col] = final_hybrid[col].round(0).astype(int)

display(final_hybrid)